<a href="https://colab.research.google.com/github/shoh0806/Capstone_Design/blob/main/Bert4Rec_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 드라이브 연결 및 설치
from google.colab import drive
drive.mount('/content/drive')
!pip install polars

# 2. 필수 라이브러리
import polars as pl
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os

# 3. 시드 고정 (재현성)
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# 1. 파일 경로 설정
path = "/content/drive/MyDrive/Capstone/data/"

# 2. 데이터 불러오기
movies = pl.read_csv(path + "movies (1).csv")
ratings = pl.read_csv(path + "ratings (1).csv")
users = pl.read_csv(path + "users (1).csv", schema_overrides={"zip": pl.String})

# 3. 데이터 결합
df_full = ratings.join(movies, on="movieId", how="left").join(users, on="userId", how="left")

# 4. 영화 ID 재번호 매기기 (0은 패딩용)
unique_items = df_full["movieId"].unique().sort().to_list()
item2idx = {item: i + 1 for i, item in enumerate(unique_items)}
num_items = len(unique_items)

df_full = df_full.with_columns(
    pl.col("movieId").replace(item2idx).alias("movieId_idx")
).sort(["userId", "timestamp"])

print(f"로드 완료! 총 아이템 수: {num_items}")

In [ ]:
# 1. 유저별 시퀀스 및 타임스탬프 추출
user_group = df_full.group_by("userId").agg([
    pl.col("movieId_idx").alias("sequence"),
    pl.col("timestamp").alias("ts_sequence")
])

# MovieLensDataset 클래스 정의
class MovieLensDataset(Dataset):
    def __init__(self, x, y, t, uids=None):
        self.x = torch.LongTensor(x)
        self.y = torch.LongTensor(y)
        self.t = torch.FloatTensor(t)
        self.uids = torch.LongTensor(uids) if uids is not None else None

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        if self.uids is not None:
            return self.x[idx], self.y[idx], self.t[idx], self.uids[idx]
        return self.x[idx], self.y[idx], self.t[idx]

# 1. 데이터를 저장할 리스트 초기화
train_X, train_y, train_T, train_Uids = [], [], [], []
val_X, val_y, val_T, val_Uids = [], [], [], []
test_X, test_y, test_T, test_Uids = [], [], [], []

maxlen = 100 # 설정된 최대 시퀀스 길이
stride = 10

# 2. 유저별로 루프를 돌며 데이터 분할 (6,040명 대상)
for user_idx in range(len(user_group)):
    user_id = user_group[user_idx, "userId"] # Get the userId
    seq = user_group[user_idx, "sequence"]
    ts_seq = user_group[user_idx, "ts_sequence"]
    n_items = len(seq)

    # 최소 3개 이상의 영화를 본 유저만 Train/Val/Test 분할이 가능합니다.
    if n_items < 20:
        continue

    seq_list = seq.to_list()
    ts_list = ts_seq.to_list()

    # (1) Test 데이터: 유저의 가장 마지막 영화 (k=1)
    target_idx = n_items - 1
    sub_seq = seq_list[:target_idx][-maxlen:]
    sub_ts = ts_list[:target_idx][-maxlen:]
    pad_len = maxlen - len(sub_seq)

    test_X.append([0] * pad_len + sub_seq)
    test_y.append(seq_list[target_idx])
    test_T.append([0.0] * pad_len + sub_ts)
    test_Uids.append(user_id) # Store userId

    # (2) Validation 데이터: 유저의 마지막에서 두 번째 영화 (k=1)
    target_idx = n_items - 2
    sub_seq = seq_list[:target_idx][-maxlen:]
    sub_ts = ts_list[:target_idx][-maxlen:]
    pad_len = maxlen - len(sub_seq)

    val_X.append([0] * pad_len + sub_seq)
    val_y.append(seq_list[target_idx])
    val_T.append([0.0] * pad_len + sub_ts)
    val_Uids.append(user_id) # Store userId

    # (3) Train 데이터: 그 이전의 모든 영화들 (Sliding Window 적용)
    indices = list(range(n_items - 3, -1, -stride))

    # 만약 유저가 영화를 어느 정도 봤는데 indices가 비어있다면,
    # 최소한 가장 최신인 n_items-3 이라도 하나 넣어줍니다.
    if not indices and n_items >= 3:
        indices = [n_items - 3]

    for i in indices:
        target_idx = i
        if target_idx < 1: continue # 타겟 이전에 최소 1개 영화는 있어야 함

        sub_seq = seq_list[:target_idx][-maxlen:]
        sub_ts = ts_list[:target_idx][-maxlen:]
        pad_len = maxlen - len(sub_seq)

        train_X.append([0] * pad_len + sub_seq)
        train_y.append(seq_list[target_idx])
        train_T.append([0.0] * pad_len + sub_ts)
        train_Uids.append(user_id) # Store userId

# 3. [중요] 수정된 출력문 (각 데이터셋의 X 리스트 길이를 확인해야 함)
print(f"데이터 준비 완료!")
print(f"Train: {len(train_X)} 개")
print(f"Val:   {len(val_X)} 개")
print(f"Test:  {len(test_X)} 개")

train_dataset = MovieLensDataset(train_X, train_y, train_T, train_Uids)
val_dataset = MovieLensDataset(val_X, val_y, val_T, val_Uids)
test_dataset = MovieLensDataset(test_X, test_y, test_T, test_Uids)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class BERT4Rec(nn.Module):
    def __init__(self, num_items, embed_dim=128, nhead=4, num_layers=4, maxlen=100, dropout=0.1):
        super(BERT4Rec, self).__init__()
        self.item_emb = nn.Embedding(num_items + 2, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(maxlen, embed_dim)
        self.emb_layernorm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=embed_dim*4,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer_blocks = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.out_layer = nn.Linear(embed_dim, num_items + 2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        batch_size, seq_len = x.size()
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0).repeat(batch_size, 1)

        e = self.dropout(self.emb_layernorm(self.item_emb(x) + self.pos_emb(pos)))
        output = self.transformer_blocks(e)

        return self.out_layer(output)

In [ ]:
def train_bert4rec(model, train_loader, num_items, epochs=10, mask_prob=0.15):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    mask_token = num_items + 1 # 마스크 토큰 번호 설정

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for bx, by, _, buids in train_loader: # by (정답 영화)를 가져옵니다.
            bx, by = bx.to(device), by.to(device)

            # [핵심] 학습 시퀀스 끝에 실제 정답(by)을 붙여서 '완성된 시퀀스'를 만듭니다.
            # 이렇게 해야 모델이 나중에 Val/Test 정답을 맞추는 법을 배웁니다.
            bx_full = torch.cat([bx[:, 1:], by.unsqueeze(1)], dim=1)

            # 완성된 시퀀스에서 랜덤 마스킹 진행
            probability_matrix = torch.full(bx_full.shape, mask_prob).to(device)
            mask_indices = torch.bernoulli(probability_matrix).bool() & (bx_full != 0)

            targets = torch.zeros_like(bx_full)
            targets[mask_indices] = bx_full[mask_indices]

            masked_bx = bx_full.clone()
            masked_bx[mask_indices] = mask_token

            optimizer.zero_grad()
            logits = model(masked_bx)
            loss = criterion(logits.view(-1, num_items + 2), targets.view(-1))

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
def get_metrics(model, loader, k=10, num_items=num_items):
    model.eval()
    hr, recall, map_score, ndcg = [], [], [], []
    mask_token = num_items + 1 # 마스크 토큰 번호

    with torch.no_grad():
        for bx, by, _, buids in loader:
            bx, by = bx.to(device), by.to(device)

            bx_masked = torch.cat([bx[:, 1:], torch.full((bx.size(0), 1), mask_token).to(device)], dim=1)

            # 모델 예측 (마지막 위치의 결과값만 추출)
            preds = model(bx_masked)[:, -1, :]

            # 0번(Padding)과 mask_token은 예측 후보에서 제외 (확률을 매우 낮게 설정)
            preds[:, 0] = -1e9
            preds[:, mask_token] = -1e9

            _, top_k = torch.topk(preds, k, dim=1)

            for target, recs in zip(by, top_k):
                recs = recs.cpu().numpy()
                target_val = target.item()

                if target_val in recs:
                    rank = np.where(recs == target_val)[0][0] + 1
                    hr.append(1); recall.append(1)
                    map_score.append(1.0 / rank); ndcg.append(1.0 / np.log2(rank + 1))
                else:
                    hr.append(0); recall.append(0); map_score.append(0); ndcg.append(0)

    return np.mean(hr), np.mean(recall), np.mean(map_score), np.mean(ndcg)
def get_global_attention(model, loader):
    model.eval()
    # 모든 레이어의 어텐션을 합치기 위한 리스트 초기화
    # 어텐션 형태: (Batch, Heads, Maxlen, Maxlen) -> 여기선 Heads를 평균냈다고 가정
    global_attn = None
    total_samples = 0

    with torch.no_grad():
        for bx, by, _ in train_loader: # by(정답)를 함께 가져옵니다
            bx, by = bx.to(device), by.to(device)

            full_seq = torch.cat([bx[:, 1:], by.unsqueeze(1)], dim=1)

            _, all_layers_attn = model(bx, return_attn=True)

            # 마지막 레이어(또는 특정 레이어)의 어텐션만 선택 (예: 마지막 레이어 = [-1])
            # 배차(Batch) 차원에 대해 평균을 내거나 합칩니다.
            last_layer_attn = all_layers_attn[-1].mean(dim=1) # (Batch, Maxlen, Maxlen)

            if global_attn is None:
                global_attn = last_layer_attn.sum(dim=0)
            else:
                global_attn += last_layer_attn.sum(dim=0)

            total_samples += bx.size(0)

    # 전체 샘플 수로 나누어 평균 어텐션 맵 생성
    avg_attn = global_attn / total_samples
    return avg_attn.cpu().numpy()

def run_experiment(layers_list=[4], epochs=30):
    results = {}
    for nl in layers_list:
        print(f"\n실험 중: Layers {nl}")
        model = BERT4Rec(num_items=num_items, num_layers=nl).to(device)
        best_hr = -1.0

        for epoch in range(epochs):
            train_bert4rec(model, train_loader, num_items, epochs=1, mask_prob=0.15)

            # 검증 및 저장 로직
            val_hr, val_recall, val_map, val_ndcg = get_metrics(model, val_loader, num_items=num_items) # 4개의 값 모두 언팩
            if epoch == 0 or val_hr > best_hr: # 첫 에폭이거나 성적이 좋아지면 저장
                best_hr = val_hr
                torch.save(model.state_dict(), f"model_L{nl}.pth")
                print(f"Epoch {epoch+1}: Best Val HR {best_hr:.4f} 저장 완료")

        model.load_state_dict(torch.load(f"model_L{nl}.pth"))
        # 4개의 지표를 모두 결과 딕셔너리에 저장
        results[nl] = get_metrics(model, test_loader, k=10)
    return results


In [ ]:
exp_results = run_experiment(layers_list=[4], epochs=30)

print("\n" + "="*75)
print(f"{'Layers':^10} | {'HR@10':^12} | {'Recall@10':^12} | {'MAP@10':^12} | {'NDCG@10':^12}")
print("-" * 75)

for nl, metrics in exp_results.items():
    hr, recall, m_score, ndcg = metrics
    print(f"{nl:^10} | {hr:^12.4f} | {recall:^12.4f} | {m_score:^12.4f} | {ndcg:^12.4f}")

print("="*75)

best 성능 가져오는 코드

In [ ]:
import json

def save_performance(metrics, filename):
    hr, recall, map_score, ndcg = metrics
    performance_data = {
        "HR@10": hr,
        "Recall@10": recall,
        "MAP@10": map_score,
        "NDCG@10": ndcg
    }
    with open(filename, 'w') as f:
        json.dump(performance_data, f, indent=4)
    print(f"Performance metrics saved to {filename}")

# run_experiment 실행 후 결과를 파일로 박제하기
exp_results = run_experiment(layers_list=[4], epochs=30)

# 메모리에 있는 수치를 JSON 파일로 저장
save_performance(exp_results[4], "final_performance_metrics.json")

In [ ]:
# 1. 모델 객체 생성 (학습할 때와 동일한 하이퍼파라미터 설정 필수)
model = BERT4Rec(num_items=num_items, num_layers=4, embed_dim=128).to(device)

# 2. 파일에서 가중치 로드
model.load_state_dict(torch.load("model_L4.pth", map_location=device))

# 3. 평가 모드로 전환 (Dropout 등을 비활성화하기 위해 필수)
model.eval()

print("최고 성능의 모델 가중치를 성공적으로 불러왔습니다.")

In [ ]:
# 1. 저장된 가중치 불러오기 (이미 하신 부분)
model.load_state_dict(torch.load("model_L4.pth", map_location=device))
model.eval()

# 2. [교체] 학습 대신 '단 한번의 평가'만 수행해서 결과 가져오기
print("모델 성능 재확인 중...")
test_metrics = get_metrics(model, test_loader, k=10) # 이 함수가 수치를 계산해줍니다

# 3. 예쁘게 출력하기
hr, recall, m_score, ndcg = test_metrics
print("\n" + "="*75)
print(f"{'Layers':^10} | {'HR@10':^12} | {'Recall@10':^12} | {'MAP@10':^12} | {'NDCG@10':^12}")
print("-" * 75)
print(f"{'4':^10} | {hr:^12.4f} | {recall:^12.4f} | {m_score:^12.4f} | {ndcg:^12.4f}")
print("="*75)

Attention weight 추출 코드

In [ ]:
from typing import Optional, Tuple

class CustomTransformerEncoderLayer(nn.TransformerEncoderLayer):
    def forward(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None, src_key_padding_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        x = src
        if self.norm_first:
            _src = self.norm1(x)
            attn_output, attn_output_weights = self.self_attn(_src, _src, _src,
                                                              attn_mask=src_mask,
                                                              key_padding_mask=src_key_padding_mask,
                                                              need_weights=True)
            x = x + self.dropout2(self.linear2(self.dropout(self.activation(self.linear1(self.norm2(x))))))
        else:
            attn_output, attn_output_weights = self.self_attn(x, x, x,
                                                              attn_mask=src_mask,
                                                              key_padding_mask=src_key_padding_mask,
                                                              need_weights=True)
            x = x + self.dropout1(attn_output)
            x = self.norm1(x)
            x = x + self.dropout2(self.linear2(self.dropout(self.activation(self.linear1(x)))))
            x = self.norm2(x)
        return x, attn_output_weights


class BERT4RecWithAttn(nn.Module):
    def __init__(self, num_items, embed_dim=128, nhead=4, num_layers=4, maxlen=100, dropout=0.1):
        super(BERT4RecWithAttn, self).__init__()
        self.item_emb = nn.Embedding(num_items + 2, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(maxlen, embed_dim)
        self.emb_layernorm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

        # 개별 레이어를 직접 정의하여 어텐션을 추출하기 쉽게 만듭니다.
        self.layers = nn.ModuleList([
            CustomTransformerEncoderLayer(d_model=embed_dim, nhead=nhead,
                                       dim_feedforward=embed_dim*4,
                                       dropout=dropout, batch_first=True,
                                       activation='gelu')
            for _ in range(num_layers)
        ])

        self.out_layer = nn.Linear(embed_dim, num_items + 2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x, return_attn=False):
        batch_size, seq_len = x.size()
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0).repeat(batch_size, 1)
        e = self.dropout(self.emb_layernorm(self.item_emb(x) + self.pos_emb(pos)))

        output = e
        all_attentions = []

        for layer in self.layers:
            if return_attn:
                output, attn_output_weights = layer(output) # Custom layer returns weights
                all_attentions.append(attn_output_weights)
            else:
                output, _ = layer(output) # No need for attention weights

        logits = self.out_layer(output)
        if return_attn:
            return logits, all_attentions
        return logits

시각화코드

In [ ]:
def get_global_attention_from_custom_model(model, loader):
    model.eval()
    # 전체 어텐션을 누적할 리스트 (레이어 개수만큼 생성)
    num_layers = len(model.layers)
    global_attn_sums = [torch.zeros((maxlen, maxlen)).to(device) for _ in range(num_layers)]
    count_matrix = torch.zeros((maxlen, maxlen)).to(device)

    print("전체 유저 어텐션 추출 중...")
    with torch.no_grad():
        for bx, _, _, _ in loader: # Unpack all 4 values from loader
            bx = bx.to(device)
            # 1. 모델에서 예측값과 모든 레이어의 어텐션 리스트를 받아옴
            _, all_attentions = model(bx, return_attn=True)

            # 2. 패딩 마스크 생성 (데이터가 있는 부분만 평균에 반영)
            valid_mask = (bx != 0).float().unsqueeze(1) * (bx != 0).float().unsqueeze(2)

            # 3. 레이어별로 어텐션 누적
            for i in range(num_layers):
                # all_attentions[i] shape: [Batch, Head, Seq, Seq] -> Head 평균 취함
                layer_attn = all_attentions[i].mean(dim=1) if all_attentions[i].dim() == 4 else all_attentions[i]
                global_attn_sums[i] += (layer_attn * valid_mask).sum(dim=0)

            count_matrix += valid_mask.sum(dim=0)

    # 4. 전체 평균 계산 (마지막 레이어 결과를 주로 사용)
    avg_attentions = [(s / (count_matrix + 1e-9)).cpu().numpy() for s in global_attn_sums]
    return avg_attentions

# 실행: 학습된 모델과 테스트 로더 사용
# global_attns = get_global_attention_from_custom_model(model, test_loader)
# last_layer_attn = global_attns[-1] # 가장 마지막 레이어의 어텐션 선택

In [ ]:
num_layers_best = 4
model_attn = BERT4RecWithAttn(num_items=num_items, embed_dim=128, nhead=4, num_layers=num_layers_best, maxlen=100).to(device)

# 2. 저장된 가중치 로드 (CustomTransformerEncoderLayer 사용에 맞게 키 이름 수정 필요)
state_dict = torch.load(f"model_L{num_layers_best}.pth")
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("transformer_blocks.layers."):
        # BERT4RecWithAttn의 layers는 직접 리스트로 관리되므로 'transformer_blocks.' 제거 후 'layers.index.' 형식으로 변경
        parts = k.split('.')
        layer_idx = parts[2] # transformer_blocks.layers.X.sub_part
        new_key = f"layers.{layer_idx}." + ".".join(parts[3:])
        new_state_dict[new_key] = v
    else:
        new_state_dict[k] = v

model_attn.load_state_dict(new_state_dict)
print(f"모델 가중치 로드 완료! (Layers: {num_layers_best})")

# 3. 전체 유저 어텐션 계산
global_avg_attns_list = get_global_attention_from_custom_model(model_attn, test_loader)
global_avg_attn = global_avg_attns_list[-1] # 마지막 레이어의 어텐션 패턴 사용
print("global_avg_attn 변수 정의 완료.")

In [ ]:
def plot_global_attention_pattern(attn_matrix, title="Global Attention Pattern"):
    plt.figure(figsize=(12, 10))

    # 상대적 시점 라벨 (최근 50개: T-49 ~ T-0)
    labels = [f"T-{i}" for i in range(maxlen-1, -1, -1)]

    sns.heatmap(attn_matrix, xticklabels=labels, yticklabels=labels, cmap='magma')

    plt.title(title)
    plt.xlabel("Key (Relative Past Positions)")
    plt.ylabel("Query (Current Context)")
    plt.xticks(rotation=90)
    plt.show()

plot_global_attention_pattern(global_avg_attn, title="BERT4Rec Global Pattern (All Users)")

In [ ]:
# 1. 필수 라이브러리 설치 (가장 안정적인 버전)
!pip install implicit

import implicit
from scipy.sparse import csr_matrix
import numpy as np

# 2. 오픈 소스에서 권장하는 "표준 데이터 변환" 방식
# 유저-아이템 행렬을 Sparse(희소) 행렬로 만드는 이 과정이 MF의 핵심입니다.
rows = df_full["userId"].to_numpy() - 1
cols = df_full["movieId_idx"].to_numpy() - 1
ratings_val = df_full["rating"].to_numpy()

# (아이템 수 x 유저 수) 형태의 CSR 행렬 생성
user_item_matrix = csr_matrix((ratings_val, (rows, cols)))

# 3. 공식 가이드에 따른 모델 선언 및 학습
# "아무 라이브러리나 대충 잡으라"고 하셨으니 가장 기본 파라미터로 설정합니다.
model = implicit.als.AlternatingLeastSquares(factors=64, iterations=15, regularization=0.1)
model.fit(user_item_matrix)

# 4. 성능(HR@10) 체크용 루프 (이게 교수님이 원하시는 '성능 확인' 부분입니다)
def check_baseline_performance(test_y, k=10):
    hits = 0
    # 모든 유저에 대해 한 번에 추천 결과 추출
    # user_item_matrix.tocsr()는 유저별 기록을 보겠다는 뜻입니다.
    user_recs, _ = model.recommend(np.arange(len(test_y)), user_item_matrix.tocsr(), N=k)

    for i, target in enumerate(test_y):
        # 정답(target)이 추천된 상위 k개 안에 있는지 확인
        if (target - 1) in user_recs[i]:
            hits += 1

    return hits / len(test_y)

# 5. 결과 출력
baseline_hr = check_baseline_performance(test_y, k=10)
print(f"\n[교수님 보고용 베이스라인 결과]")
print(f"Matrix Factorization (ALS) 성능: {baseline_hr:.4f}")

In [ ]:
# 1. 라이브러리 설치 (NumPy 2.0과 호환되는 최신 라이브러리)
!pip install implicit

In [ ]:
import implicit
from scipy.sparse import csr_matrix
import numpy as np

# 1. 데이터 준비 (Polars -> Pandas 변환 및 명시적 매핑)
pdf = df_full.to_pandas()
u_map = {id: i for i, id in enumerate(pdf["userId"].unique())}
i_map = {id: i for i, id in enumerate(pdf["movieId_idx"].unique())}

# 2. 행렬 생성 (User x Item 방향으로 고정)
# ★ 핵심: shape를 (유저 수, 아이템 수)로 설정하여 모델이 유저를 정확히 인식하게 합니다.
u_train = pdf["userId"].map(u_map).values
i_train = pdf["movieId_idx"].map(i_map).values
r_train = pdf["rating"].values.astype(np.float32)

# 행(User), 열(Item) 순서입니다.
train_matrix = csr_matrix((r_train, (u_train, i_train)),
                          shape=(len(u_map), len(i_map)))

# 3. 모델 학습 (CPU 모드 사용)
# use_gpu=False를 설정하여 인덱스 에러가 잦은 GPU 커널을 피합니다.
model = implicit.als.AlternatingLeastSquares(factors=64, iterations=20, use_gpu=False, random_state=42)

# ★ 중요: train_matrix를 그대로 넣습니다 (T.tocsr() 하지 마세요)
model.fit(train_matrix)

# 4. 성능 평가 (Hit Rate @ 10)
def calculate_final_hr(test_y_list, k=10):
    hits = 0
    # 평가 대상 유저들의 ID를 순서대로 가져옵니다.
    valid_u_ids = [uid for uid, seq in zip(user_group["userId"], user_group["sequence"]) if len(seq) >= 3]

    print(f"MF 지표 계산 중 (대상: {len(test_y_list)}명)...")

    # 모델에서 추천 결과 추출
    # 유저 인덱스를 배열로 전달하여 한 번에 추천을 받습니다.
    u_indices = np.array([u_map[uid] for uid in valid_u_ids])
    ids, _ = model.recommend(u_indices, train_matrix[u_indices], N=k, filter_already_liked_items=False)

    for i in range(len(test_y_list)):
        target_real_id = test_y_list[i]
        if target_real_id not in i_map: continue

        target_i_idx = i_map[target_real_id]

        # 추천된 10개 안에 정답이 있는지 확인
        if target_i_idx in ids[i]:
            hits += 1

    return hits / len(test_y_list)

# 5. 최종 수치 확인
mf_hr = calculate_final_hr(test_y, k=10)

print("\n" + "="*45)
print(f"MF 베이스라인 HR@10: {mf_hr:.4f}")
print("="*45)

연도 및 장르 추출

In [ ]:
import re

# 영화 연도 추출 함수
def extract_year(title):
    match = re.search(r'\((\d{4})\)', title)
    return int(match.group(1)) if match else 1990 # 기본값

# 메타데이터 정리
pdf_movies = movies.to_pandas()
pdf_movies['movieId_idx'] = pdf_movies['movieId'].map(item2idx)
pdf_movies['year'] = pdf_movies['title'].apply(extract_year)
pdf_movies['is_old'] = pdf_movies['year'] < 1990 # 90년대 이전 영화 기준

# 아이템 인덱스별 메타데이터 매핑용 딕셔너리
idx2year = {item2idx[row['movieId']]: row['year'] for _, row in pdf_movies.iterrows() if row['movieId'] in item2idx}
idx2genres = {item2idx[row['movieId']]: row['genres'].split('|') for _, row in pdf_movies.iterrows() if row['movieId'] in item2idx}
idx2is_old = {item2idx[row['movieId']]: row['is_old'] for _, row in pdf_movies.iterrows() if row['movieId'] in item2idx}

마지막 시점이 과거 시퀀스의 어떤 특징에 어텐션을 주었는지 계산, 맞춘 유저와 못 맞춘 유저 구분해서 분석

In [ ]:
import pandas as pd

def analyze_attention_depth(model, loader, k=10):
    model.eval()
    user_analysis = []

    print("유저별 어텐션 심층 분석 중...")
    with torch.no_grad():
        for bx, by, _, uids in loader: # Unpack uids
            bx, by = bx.to(device), by.to(device)
            logits, all_attentions = model(bx, return_attn=True)

            # 마지막 레이어의 어텐션 [Batch, Seq, Seq] (MultiHeadAttention이 이미 Head 평균)
            attn = all_attentions[-1]
            last_attn = attn[:, -1, :] # 마지막 예측 시점의 어텐션 [Batch, Seq]

            # 예측 성공 여부 확인
            mask_token = num_items + 1
            preds = logits[:, -1, :].clone()
            preds[:, 0] = -1e9
            preds[:, mask_token] = -1e9
            _, top_k = torch.topk(preds, k, dim=1)

            for i in range(bx.size(0)):
                u_seq = bx[i].cpu().numpy()
                u_attn = last_attn[i].cpu().numpy()
                is_hit = by[i].item() in top_k[i].cpu().numpy()
                user_id = uids[i].item() # Get userId

                # 유효한(패딩이 아닌) 인덱스만 추출
                valid_idx = np.where(u_seq != 0)[0]
                if len(valid_idx) == 0: continue

                # 1. 시퀀스 위치별 어텐션 (Recency 분석)
                # T-1(가장 최근) ~ T-100(가장 과거)
                recency_attn = u_attn[valid_idx][-5:].sum() # 최근 5개에 준 어텐션
                distant_attn = u_attn[valid_idx][:-10].sum() # 10개보다 더 과거에 준 어텐션

                # 2. 메타데이터별 어텐션
                genre_hits = {}
                old_movie_attn = 0

                for idx_in_seq in valid_idx:
                    item_idx = u_seq[idx_in_seq]
                    weight = u_attn[idx_in_seq]

                    # 올드 영화 어텐션 합산
                    if idx2is_old.get(item_idx, False):
                        old_movie_attn += weight

                    # 장르별 어텐션 합산
                    for g in idx2genres.get(item_idx, []):
                        genre_hits[g] = genre_hits.get(g, 0) + weight

                # 유저별 데이터 저장
                user_analysis.append({
                    'userId': user_id, # Add userId
                    'is_hit': is_hit,
                    'recency_ratio': recency_attn,
                    'distant_ratio': distant_attn,
                    'old_movie_attn': old_movie_attn,
                    'top_genre': max(genre_hits, key=genre_hits.get) if genre_hits else None,
                    'attn_map': u_attn[valid_idx]
                })

    return pd.DataFrame(user_analysis)

# 분석 실행
analysis_df = analyze_attention_depth(model_attn, test_loader)

최근 아이템보다 예전 아이템에 어텐션을 더 많이 준 유저들 분석

In [ ]:
# 최근보다 과거에 더 집중하는 '장기 기억' 유저 그룹
long_term_users = analysis_df[analysis_df['distant_ratio'] > analysis_df['recency_ratio']]

print(f"장기 기억 의존 유저 비율: {len(long_term_users)/len(analysis_df)*100:.2f}%")
print(f"이들의 Hit Rate: {long_term_users['is_hit'].mean():.4f}")

# 이 유저들의 주력 장르 확인
print("과거에 집착하는 유저들의 선호 장르 TOP 5:")
print(long_term_users['top_genre'].value_counts().head(5))

취향이 투렷하거나 특정 연도 영화를 선호하는 유저의 특성

In [ ]:
# 올드 영화 어텐션 비중이 높은 유저 (상위 20%)
oldies_lovers = analysis_df[analysis_df['old_movie_attn'] > analysis_df['old_movie_attn'].quantile(0.8)]

print(f"올드 무비 팬 그룹의 Hit Rate: {oldies_lovers['is_hit'].mean():.4f}")

정답을 맞춘 애들과 못 맞춘 애들의 어텐션을 주는 패턴이 어떻게 다른지 분석

In [ ]:
plt.figure(figsize=(10, 6))

# 맞춘 유저들의 평균 어텐션 분포 (뒤에서부터 역순)
hit_attn = np.mean([x[-20:] for x in analysis_df[analysis_df['is_hit']==True]['attn_map'] if len(x)>=20], axis=0)
miss_attn = np.mean([x[-20:] for x in analysis_df[analysis_df['is_hit']==False]['attn_map'] if len(x)>=20], axis=0)

plt.plot(hit_attn, label='Hits (Correct)', marker='o')
plt.plot(miss_attn, label='Misses (Incorrect)', marker='x')
plt.title("Attention Weight Distribution: Hits vs Misses (Last 20 items)")
plt.xlabel("Relative Position (Recent to Older)")
plt.ylabel("Average Attention Weight")
plt.legend()
plt.show()

모델이 특정 영화가 어텐션을 꽂았는지 혹은 여러 영화에 분산했는지 분석

In [ ]:
import scipy.stats

def analyze_entropy_and_focus(analysis_df):
    # 각 유저의 어텐션 분포에 대한 엔트로피 계산
    analysis_df['entropy'] = analysis_df['attn_map'].apply(lambda x: scipy.stats.entropy(x))

    # 1. 엔트로피와 정답률의 관계
    entropy_hit = analysis_df.groupby('is_hit')['entropy'].mean()

    # 2. "결단력 있는 유저" vs "혼란스러운 유저" 그룹화
    analysis_df['focus_type'] = pd.qcut(analysis_df['entropy'], q=3, labels=['Decisive', 'Neutral', 'Confused'])

    print("--- [인사이트 1: 모델의 확신도 분석] ---")
    print(analysis_df.groupby('focus_type')['is_hit'].mean())
    print("\n* 해석: Confused 그룹의 정답률이 낮다면, 유저의 취향이 너무 파편화되어 모델이 맥락을 못 잡고 있음을 의미함.")
    return analysis_df

analysis_df = analyze_entropy_and_focus(analysis_df)

# Create user_groups dictionary based on analysis_df
user_groups = {}
for idx, row in analysis_df.iterrows():
    if row['focus_type'] == 'Decisive':
        user_groups[row['userId']] = 'Q1'
    elif row['focus_type'] == 'Confused':
        user_groups[row['userId']] = 'Q4'

가까이를 안보는 유저들의 유형이 나뉘는지 확인

In [ ]:
from sklearn.cluster import KMeans

def cluster_distant_focus_users(analysis_df):
    # '가까이를 안 보는 유저' (최근 5개 어텐션 합이 하위 30%인 유저) 필터링
    distant_users = analysis_df[analysis_df['recency_ratio'] < analysis_df['recency_ratio'].quantile(0.3)].copy()

    # 어텐션 맵의 길이를 맞추기 위해 마지막 20개만 사용
    features = np.array([x[-20:] if len(x)>=20 else np.pad(x, (20-len(x), 0)) for x in distant_users['attn_map']])

    # 3개 유형으로 클러스터링
    kmeans = KMeans(n_clusters=3, random_state=42).fit(features)
    distant_users['cluster'] = kmeans.labels_

    print("--- [인사이트 2: 근거리 무시 유저의 세부 유형] ---")
    for i in range(3):
        c_group = distant_users[distant_users['cluster'] == i]
        print(f"유형 {i}: 평균 Hit Rate {c_group['is_hit'].mean():.4f}, 주력 장르: {c_group['top_genre'].mode()[0]}")

    print("\n* 해석: 유형 0은 특정 과거 장르에 집착, 유형 1은 주기적 패턴 등 '가까이를 안 봐도 맞추는 근거'를 찾아낼 수 있음.")
    return distant_users

distant_clusters = cluster_distant_focus_users(analysis_df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_entropy_analysis(analysis_df):
    plt.figure(figsize=(10, 6))

    # 1. 엔트로피 그룹별 Hit Rate 시각화
    ax = sns.barplot(x='focus_type', y='is_hit', data=analysis_df, palette='viridis', order=['Decisive', 'Neutral', 'Confused'])

    # 평균선 추가
    plt.axhline(analysis_df['is_hit'].mean(), color='red', linestyle='--', label='Global Average HR')

    plt.title("Hit Rate by Model's Decision Confidence (Entropy)", fontsize=15)
    plt.xlabel("User Type (Based on Attention Focus)", fontsize=12)
    plt.ylabel("Hit Rate (HR@10)", fontsize=12)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

plot_entropy_analysis(analysis_df)

근거리 무시 유저의 세부 유형 시각화

In [ ]:
def plot_distant_clusters(distant_clusters):
    plt.figure(figsize=(12, 6))

    for i in range(3):
        # 각 클러스터의 평균 어텐션 패턴 계산
        cluster_data = distant_clusters[distant_clusters['cluster'] == i]
        mean_pattern = np.mean([x[-30:] for x in cluster_data['attn_map'] if len(x)>=30], axis=0)

        plt.plot(mean_pattern, label=f'Cluster {i} (HR: {cluster_data["is_hit"].mean():.2%})', lw=2)

    plt.title("Attention Patterns of 'Distant-Focus' User Groups", fontsize=15)
    plt.xlabel("Relative Position (Older <--- Recent)", fontsize=12)
    plt.ylabel("Mean Attention Weight", fontsize=12)
    plt.xticks(range(30), [f"T-{29-i}" for i in range(30)], rotation=90)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_distant_clusters(distant_clusters)

유저별 시퀀스 리스트, 인기도 데이터

In [ ]:
# 분석을 위한 유저별 시퀀스 리스트 추출 (test_loader와 순서 동일)
u_seq_list = [seq.to_list()[-maxlen:] for uid, seq in zip(user_group["userId"], user_group["sequence"]) if len(seq) >= 3]

# 영화별 인기도(노출 빈도) 계산
item_popularity = pdf['movieId_idx'].value_counts().to_dict()

모델이 대중적인 영화에 의존하는지 확인

In [ ]:
# 데이터 계산
pop_weights = []
for i, row in analysis_df.iterrows():
    seq = u_seq_list[i]
    attn = row['attn_map']
    # 어텐션이 실린 아이템들의 인기도 가중 평균
    weighted_pop = sum([item_popularity.get(it, 0) * att for it, att in zip(seq, attn)])
    pop_weights.append(weighted_pop)

analysis_df['attn_to_popular'] = pop_weights

# 시각화 함수 호출
import seaborn as sns
import matplotlib.pyplot as plt

def plot_popularity_bias(df):
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='is_hit', y='attn_to_popular', data=df, palette='coolwarm')
    plt.title('Attention to Popular Items by Hit Status')
    plt.xlabel('Is Hit (True/False)')
    plt.ylabel('Weighted Popularity Score')
    plt.xticks([0, 1], ['Miss', 'Hit'])
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

plot_popularity_bias(analysis_df)

시간이 지날수록 모델의 관심도가 어떻게 변하는지 확인

In [ ]:
# 데이터 계산 (Time Gap 데이터 생성)
gaps_vs_attn = []
for i, (seq, ts) in enumerate(zip(user_group["sequence"], user_group["ts_sequence"])):
    if i >= len(analysis_df): break

    valid_attn = analysis_df.iloc[i]['attn_map']
    ts_list = ts.to_list()[-len(valid_attn):]
    last_ts = ts_list[-1]

    for t, a in zip(ts_list[:-1], valid_attn[:-1]):
        gap_days = (last_ts - t) / (3600 * 24)
        gaps_vs_attn.append([gap_days, a, analysis_df.iloc[i]['is_hit']])

gap_df = pd.DataFrame(gaps_vs_attn, columns=['days_passed', 'attn_weight', 'is_hit'])
gap_df['time_bin'] = pd.cut(gap_df['days_passed'], bins=[0, 1, 7, 30, 180, 5000], labels=['1d', '1w', '1m', '6m', 'Long'])

import seaborn as sns
import matplotlib.pyplot as plt

def plot_temporal_decay(df):
    plt.figure(figsize=(10, 6))
    sns.barplot(x='time_bin', y='attn_weight', hue='is_hit', data=df, errorbar=None, palette='viridis')
    plt.title('Average Attention Weight by Time Gap and Hit Status')
    plt.xlabel('Time Gap')
    plt.ylabel('Average Attention Weight')
    plt.legend(title='Is Hit')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

# 시각화 함수 호출
plot_temporal_decay(gap_df)

모델이 중요하게 생각하는 영화 TOP 10을 시각화

In [ ]:
# 데이터 계산 (Anchor Item 점수 산출)
item_attn_total = {}
item_count = {}

for i, row in analysis_df.iterrows():
    seq = u_seq_list[i]
    attn = row['attn_map']
    for it, a in zip(seq, attn):
        item_attn_total[it] = item_attn_total.get(it, 0) + a
        item_count[it] = item_count.get(it, 0) + 1

anchor_scores = {it: item_attn_total[it]/item_count[it] for it in item_attn_total if item_count[it] > 10}
sorted_anchors = sorted(anchor_scores.items(), key=lambda x: x[1], reverse=True)[:10]

# 시각화 함수 정의
def plot_anchor_items(sorted_anchors, pdf_movies):
    items = [item[0] for item in sorted_anchors]
    scores = [item[1] for item in sorted_anchors]

    item_titles = []
    for item_idx in items:
        title_row = pdf_movies[pdf_movies['movieId_idx'] == item_idx]
        if not title_row.empty:
            item_titles.append(title_row['title'].iloc[0])
        else:
            item_titles.append(f"Unknown Item {item_idx}")

    plt.figure(figsize=(12, 7))
    sns.barplot(x=scores, y=item_titles, hue=item_titles, palette='viridis', legend=False)
    plt.title('Top 10 Anchor Items by Average Attention Score', fontsize=16)
    plt.xlabel('Average Attention Score', fontsize=12)
    plt.ylabel('Movie Title', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# 시각화 함수 호출
plot_anchor_items(sorted_anchors, pdf_movies)

모델이 유저의 과거 영화와 최근 영화 중 어디에 비중을 두는지 비교

In [ ]:
def analyze_primacy_recency(analysis_df):
    # 초두/최신 효과 분석
    # 초두 효과 (Primacy): 시퀀스 앞부분에 대한 어텐션
    # 최신 효과 (Recency): 시퀀스 뒷부분에 대한 어텐션

    # 시퀀스의 처음 5개 아이템에 대한 평균 어텐션
    analysis_df['primacy_attn'] = analysis_df['attn_map'].apply(
        lambda x: np.mean(x[:5]) if len(x) >= 5 else 0
    )
    # 시퀀스의 마지막 5개 아이템에 대한 평균 어텐션 (test_y를 예측하므로 마지막 어텐션이 중요)
    analysis_df['recency_attn_last_5'] = analysis_df['attn_map'].apply(
        lambda x: np.mean(x[-5:]) if len(x) >= 5 else 0
    )

    print("--- [인사이트 6: 초두 효과 vs 최신 효과] ---")
    print("Hit 유저의 Primacy Attention:", analysis_df[analysis_df['is_hit']]['primacy_attn'].mean())
    print("Miss 유저의 Primacy Attention:", analysis_df[~analysis_df['is_hit']]['primacy_attn'].mean())
    print("Hit 유저의 Recency Attention (Last 5):", analysis_df[analysis_df['is_hit']]['recency_attn_last_5'].mean())
    print("Miss 유저의 Recency Attention (Last 5):", analysis_df[~analysis_df['is_hit']]['recency_attn_last_5'].mean())

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(x='is_hit', y='primacy_attn', data=analysis_df)
    plt.title('Primacy Attention by Hit Status')
    plt.xlabel('Is Hit')
    plt.ylabel('Average Primacy Attention')

    plt.subplot(1, 2, 2)
    sns.boxplot(x='is_hit', y='recency_attn_last_5', data=analysis_df)
    plt.title('Recency Attention (Last 5) by Hit Status')
    plt.xlabel('Is Hit')
    plt.ylabel('Average Recency Attention')
    plt.tight_layout()
    plt.show()

유저가 얼마나 다양한 장르를 시청했는지 모델의 어텐션 엔트로피에 어떤 영향을 주었는지

In [ ]:
def analyze_genre_diversity(analysis_df, u_seq_list):
    # 유저 시퀀스의 장르 다양성 계산
    genre_diversity = []
    for i, seq in enumerate(u_seq_list):
        user_genres = set()
        for item_idx in seq:
            user_genres.update(idx2genres.get(item_idx, []))
        genre_diversity.append(len(user_genres))
    analysis_df['genre_diversity'] = genre_diversity

    print("--- [인사이트 7: 장르 다양성 vs 모델 혼란도] ---")
    # 장르 다양성 구간별 Hit Rate
    analysis_df['diversity_bin'] = pd.qcut(analysis_df['genre_diversity'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    print(analysis_df.groupby('diversity_bin')['is_hit'].mean())

    plt.figure(figsize=(8, 6))
    sns.barplot(x='diversity_bin', y='is_hit', data=analysis_df, palette='GnBu_r')
    plt.title('Hit Rate by Genre Diversity Quartile')
    plt.xlabel('Genre Diversity Quartile')
    plt.ylabel('Hit Rate')
    plt.show()

어떤 장르가 시퀀스에 등장했을 때 모델의 어텐션을 독점하는지 확인

In [ ]:
def analyze_genre_power(analysis_df, u_seq_list):
    # 각 장르가 얻는 총 어텐션 합계 계산
    genre_total_attn = {}
    genre_count = {}

    for i, row in analysis_df.iterrows():
        seq = u_seq_list[i]
        attn = row['attn_map']

        for item_idx, attn_weight in zip(seq, attn):
            genres = idx2genres.get(item_idx, [])
            for g in genres:
                genre_total_attn[g] = genre_total_attn.get(g, 0) + attn_weight
                genre_count[g] = genre_count.get(g, 0) + 1

    # 평균 어텐션이 높은 장르 TOP 10
    genre_avg_attn = {g: genre_total_attn[g] / genre_count[g] for g in genre_total_attn if genre_count[g] > 50} # 최소 50개 이상 등장한 장르만 고려
    sorted_genres = sorted(genre_avg_attn.items(), key=lambda x: x[1], reverse=True)[:10]

    print("--- [인사이트 8: 장르별 '어텐션 탈취력'] ---")
    genres = [g[0] for g in sorted_genres]
    attns = [g[1] for g in sorted_genres]

    plt.figure(figsize=(10, 6))
    sns.barplot(x=attns, y=genres, palette='flare')
    plt.title('Top 10 Genres by Average Attention Weight')
    plt.xlabel('Average Attention Weight')
    plt.ylabel('Genre')
    plt.show()

전체 유저의 100개 위치에 대한 평균 어텐션을 히트맵으로 시각화

In [ ]:
def plot_positional_heatmap(analysis_df):
    # 어텐션 맵의 최대 길이에 맞춰 패딩
    padded_attns_hit = []
    padded_attns_miss = []
    max_len_attn = 0

    for amap in analysis_df[analysis_df['is_hit']]['attn_map']:
        max_len_attn = max(max_len_attn, len(amap))
    for amap in analysis_df[~analysis_df['is_hit']]['attn_map']:
        max_len_attn = max(max_len_attn, len(amap))

    for amap in analysis_df[analysis_df['is_hit']]['attn_map']:
        padded_attns_hit.append(np.pad(amap, (max_len_attn - len(amap), 0), 'constant'))
    for amap in analysis_df[~analysis_df['is_hit']]['attn_map']:
        padded_attns_miss.append(np.pad(amap, (max_len_attn - len(amap), 0), 'constant'))

    avg_attn_hit = np.mean(padded_attns_hit, axis=0)
    avg_attn_miss = np.mean(padded_attns_miss, axis=0)

    # 시각화를 위해 길이를 maxlen으로 제한 (너무 길면 보기 힘듦)
    plot_len = min(max_len_attn, maxlen) # maxlen은 이미 100으로 정의됨

    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    sns.heatmap(avg_attn_hit[-plot_len:].reshape(1, -1), cmap='viridis', cbar=False)
    plt.title('Average Positional Attention (Hits)')
    plt.xlabel('Relative Position (Older -> Recent)')
    plt.yticks([]) # y축 라벨 제거

    plt.subplot(1, 2, 2)
    sns.heatmap(avg_attn_miss[-plot_len:].reshape(1, -1), cmap='viridis', cbar=False)
    plt.title('Average Positional Attention (Misses)')
    plt.xlabel('Relative Position (Older -> Recent)')
    plt.yticks([]) # y축 라벨 제거
    plt.tight_layout()
    plt.show()

In [ ]:
# [실행 전 체크] 데이터가 준비되었는지 확인
if 'u_seq_list' not in locals():
    u_seq_list = [seq.to_list()[-maxlen:] for uid, seq in zip(user_group["userId"], user_group["sequence"]) if len(seq) >= 3]

# ---------------------------------------------------------
# 6. 초두 효과 vs 최신 효과 (Primacy vs Recency) 실행
# ---------------------------------------------------------
print("\n[분석 6] 초두/최신 효과 분석 중...")
analyze_primacy_recency(analysis_df)

# ---------------------------------------------------------
# 7. 장르 다양성 vs 모델 혼란도 실행
# ---------------------------------------------------------
print("\n[분석 7] 장르 다양성 상관관계 분석 중...")
analyze_genre_diversity(analysis_df, u_seq_list)

# ---------------------------------------------------------
# 8. 장르별 '어텐션 탈취력' 실행
# ---------------------------------------------------------
print("\n[분석 8] 장르별 어텐션 탈취력 분석 중...")
analyze_genre_power(analysis_df, u_seq_list)

# ---------------------------------------------------------
# 9. 위치별 어텐션 히트맵 (Global Position) 실행
# ---------------------------------------------------------
print("\n[분석 9] 전체 위치별 어텐션 패턴 시각화 중...")
plot_positional_heatmap(analysis_df)

유저의 먼 과거 영화와 가까운 과거 영화 중 모델이 어디에 더 비중을 두는지 확인

In [ ]:
def analyze_primacy_recency(analysis_df):
    # 첫 5개(초두) vs 마지막 5개(최신) 어텐션 비중 계산
    analysis_df['primacy_weight'] = analysis_df['attn_map'].apply(lambda x: x[:5].sum())
    analysis_df['recency_weight'] = analysis_df['attn_map'].apply(lambda x: x[-5:].sum())

    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=analysis_df, x='primacy_weight', y='recency_weight', hue='is_hit', alpha=0.5)
    plt.title("Primacy vs Recency Attention Bias")
    plt.xlabel("Attention on First 5 Items (Oldest)")
    plt.ylabel("Attention on Last 5 Items (Newest)")
    plt.show()

    print(f"초두 효과 의존 유저 평균 정답률: {analysis_df[analysis_df['primacy_weight'] > analysis_df['recency_weight']]['is_hit'].mean():.4f}")
    print(f"최신 효과 의존 유저 평균 정답률: {analysis_df[analysis_df['recency_weight'] > analysis_df['primacy_weight']]['is_hit'].mean():.4f}")

analyze_primacy_recency(analysis_df)

셔플링 코드

In [ ]:
import numpy as np
import pandas as pd
import torch

def get_attention_map(model, seq):
    model.eval()
    input_seq_tensor = torch.LongTensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():
        _, all_attentions = model(input_seq_tensor, return_attn=True)

    last_layer_attn = all_attentions[-1]

    # last_layer_attn already has shape (Batch, SeqLen, SeqLen) where heads are implicitly averaged.
    # No need to call .mean(dim=1) again, as it would collapse the sequence dimension.

    valid_len = (input_seq_tensor[0] != 0).sum().item()

    if valid_len == 0:
        return np.zeros(maxlen)

    # Attention of the last valid item (query) to all valid items (keys)
    # last_layer_attn[batch_idx, query_pos, key_pos]
    attn_vector_full_len = last_layer_attn[0, valid_len - 1, :].cpu().numpy()
    return attn_vector_full_len

def perform_professor_tests(model, sample_seq):

    model.eval()

    # [실험 A] 원본 데이터
    orig_attn = get_attention_map(model, sample_seq)

    # [실험 B] 전체 순서 셔플
    shuffled_seq = sample_seq.copy()
    valid_len = (np.array(shuffled_seq) != 0).sum()
    if valid_len > 0:
        non_padded_part = shuffled_seq[maxlen - valid_len:]
        np.random.shuffle(non_padded_part)
        shuffled_seq[maxlen - valid_len:] = non_padded_part
    shuffled_attn = get_attention_map(model, shuffled_seq)

    # [실험 C] 이상치(Noise) 삽입 (예: 50번 위치에 다른 장르 영화 강제 주입)
    noise_seq = sample_seq.copy()
    valid_len_noise = (np.array(noise_seq) != 0).sum()
    if valid_len_noise > 0:
        # Insert noise at a position within the valid sequence length (e.g., 50th position from end or earliest valid)
        noise_pos = max(maxlen - valid_len_noise, maxlen - 50)
        # Ensure 9999 is a valid item ID or mask token, assuming it's an out-of-vocab item for testing
        noise_seq[noise_pos] = 9999
    noise_attn = get_attention_map(model, noise_seq)

    return orig_attn, shuffled_attn, noise_attn

In [ ]:
def calculate_hr(logits, target, k=10):
    """Calculates Hit Rate @ k for a batch."""
    mask_token = num_items + 1 # Assuming num_items is globally available
    preds = logits[:, -1, :].clone()
    preds[:, 0] = -1e9 # Exclude padding
    preds[:, mask_token] = -1e9 # Exclude mask token
    _, top_k = torch.topk(preds, k, dim=1)

    hits = 0
    for i in range(target.size(0)):
        if target[i].item() in top_k[i].cpu().numpy():
            hits += 1
    return hits / target.size(0) if target.size(0) > 0 else 0

def evaluate_robustness(model, test_loader, device):
    """모든 유저를 대상으로 셔플링 시 성능 하락폭 계산 (표 작성용)"""
    model.eval()
    orig_hits, shuffle_hits = [], []

    with torch.no_grad():
        for seq, target, _, uids in test_loader: # Corrected unpacking
            seq = seq.to(device)
            target = target.to(device)

            # 1. 원본 데이터 성능
            logits = model(seq)
            orig_hits.append(calculate_hr(logits, target))

            # 2. 전체 셔플 데이터 성능
            shuffled_seq = seq.clone()
            for i in range(seq.size(0)): # Iterate through batch
                current_valid_len = (seq[i] != 0).sum().item()
                if current_valid_len > 0:
                    non_padded_part = shuffled_seq[i, maxlen - current_valid_len:].clone()
                    shuffled_indices = torch.randperm(current_valid_len)
                    shuffled_seq[i, maxlen - current_valid_len:] = non_padded_part[shuffled_indices]

            s_logits = model(shuffled_seq)
            shuffle_hits.append(calculate_hr(s_logits, target))

    print(f"원본 데이터 HR@10: {np.mean(orig_hits):.4f}")
    print(f"셔플 데이터 HR@10: {np.mean(shuffle_hits):.4f}")
    print(f"하락폭: {(np.mean(orig_hits) - np.mean(shuffle_hits)) / np.mean(orig_hits) * 100:.2f}%")

In [ ]:
def plot_professor_check(orig_attn, shuffled_attn, orig_seq, shuffled_seq):
    """아이템이 이동해도 어텐션이 따라가는지 확인하는 시각화"""
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))

    # 원본 어텐션 히트맵
    sns.heatmap(orig_attn.reshape(1, -1), ax=axes[0], cmap='YlGnBu')
    axes[0].set_title("Original Sequence Attention")

    # 셔플 어텐션 히트맵
    sns.heatmap(shuffled_attn.reshape(1, -1), ax=axes[1], cmap='YlGnBu')
    axes[1].set_title("Shuffled Sequence Attention")

    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_diverse_robustness(model, test_loader, device, k=10):
    """전체 유저를 대상으로 4가지 셔플링 시나리오 성능 측정"""
    model.eval()
    results = {
        'Original': [],
        'Full_Shuffle': [],
        'Recent_10_Shuffle': [],
        'History_90_Shuffle': []
    }

    print("전체 유저 대상 강건성 테스트 중... (배치 단위 처리)")

    with torch.no_grad():
        for seq, target, _, uids in test_loader: # Corrected unpacking
            seq = seq.to(device)
            target = target.to(device)
            batch_size = seq.size(0)

            # [A] 원본 데이터 성능
            results['Original'].append(calculate_hr(model(seq), target, k))

            # [B] 전체 셔플 (100개 모두)
            full_shuffled = seq.clone()
            for i in range(batch_size):
                valid_len = (seq[i] != 0).sum().item()
                if valid_len > 0:
                    non_padded_part = full_shuffled[i, maxlen - valid_len:].clone()
                    shuffled_indices = torch.randperm(valid_len)
                    full_shuffled[i, maxlen - valid_len:] = non_padded_part[shuffled_indices]
            results['Full_Shuffle'].append(calculate_hr(model(full_shuffled), target, k))

            # [C] 최근 10개만 셔플 (최신 맥락 파괴)
            recent_shuffled = seq.clone()
            for i in range(batch_size):
                valid_len = (seq[i] != 0).sum().item()
                if valid_len >= 10:
                    target_start_idx = maxlen - 10
                    actual_start_of_10 = max(maxlen - valid_len, target_start_idx)
                    items_to_shuffle = recent_shuffled[i, actual_start_of_10:].clone()
                    shuffled_indices = torch.randperm(items_to_shuffle.size(0))
                    recent_shuffled[i, actual_start_of_10:] = items_to_shuffle[shuffled_indices]
                elif valid_len > 0: # If less than 10 valid items, shuffle all valid items
                    items_to_shuffle = recent_shuffled[i, maxlen - valid_len:].clone()
                    shuffled_indices = torch.randperm(items_to_shuffle.size(0))
                    recent_shuffled[i, maxlen - valid_len:] = items_to_shuffle[shuffled_indices]
            results['Recent_10_Shuffle'].append(calculate_hr(model(recent_shuffled), target, k))

            # [D] 과거 90개만 셔플 (장기 맥락 파괴)
            history_shuffled = seq.clone()
            for i in range(batch_size):
                valid_len = (seq[i] != 0).sum().item()
                if valid_len > 10:
                    start_history = maxlen - valid_len
                    end_history = maxlen - 10
                    items_to_shuffle = history_shuffled[i, start_history:end_history].clone()
                    shuffled_indices = torch.randperm(items_to_shuffle.size(0))
                    history_shuffled[i, start_history:end_history] = items_to_shuffle[shuffled_indices]
            results['History_90_Shuffle'].append(calculate_hr(model(history_shuffled), target, k))

    # Final result summarization
    summary = []
    for key, values in results.items():
        avg_hr = np.mean(values)
        summary.append({'Scenario': key, 'HR@10': avg_hr})

    df = pd.DataFrame(summary)
    base_hr = df.loc[df['Scenario'] == 'Original', 'HR@10'].values[0]
    df['Drop %'] = df['HR@10'].apply(lambda x: ((base_hr - x) / base_hr * 100) if base_hr > 0 else 0)

    return df

In [ ]:
# --- [전체 유저 대상 최종 실험 실행] ---
print("="*60)
print("   [Final Report] 데이터 변형에 따른 전체 유저 성능 변화")
print("="*60)

# 1. 4가지 셔플링 시나리오 실행
robustness_table = evaluate_diverse_robustness(model_attn, test_loader, device, k=10)

# 2. 결과 출력 (딱딱하게 정리된 표)
print("\n" + robustness_table.to_string(index=False))

print("-" * 60)
print(f"가장 치명적인 변형: {robustness_table.loc[robustness_table['Drop %'].idxmax(), 'Scenario']}")
print("="*60)

그룹별 민감도 분석

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_noise_sensitivity_robust(model, test_loader, user_groups, device):
    model.eval()

    # 1. 안전한 노이즈 아이템 선정 (데이터셋에 실제 존재하는 가장 작은 ID)
    # 보통 0은 패딩이므로 1번을 사용하거나, 데이터셋 내 실존 ID를 사용합니다.
    noise_item_id = 1

    results = {'Q1': {'orig_hr': [], 'noise_hr': [], 'noise_attn': []},
               'Q4': {'orig_hr': [], 'noise_hr': [], 'noise_attn': []}}

    print("--- [분석 시작] 그룹별 노이즈 주입 테스트 중... ---")

    with torch.no_grad():
        for batch in test_loader:
            # 에러 로그에 따라 4개 값(seq, target, ts, uids)을 언패킹합니다.
            seq_batch, target_batch, _, uids = batch

            seq_batch = seq_batch.to(device)
            target_batch = target_batch.to(device)

            # 배치 내 각 유저별 처리
            for i in range(seq_batch.size(0)):
                uid = uids[i].item()
                group = user_groups.get(uid)

                # Q1(확고), Q4(잡식) 그룹이 아니면 스킵
                if group not in ['Q1', 'Q4']:
                    continue

                # [A] 원본 시퀀스 성능 측정
                orig_logits = model(seq_batch[i:i+1])
                results[group]['orig_hr'].append(calculate_hr(orig_logits, target_batch[i:i+1]))

                # [B] 노이즈 시퀀스 생성 (최근 5번째 위치 T-5에 노이즈 삽입)
                noise_seq = seq_batch[i:i+1].clone()
                noise_seq[0, -1] = noise_item_id # 맨 마지막 위치 (T-1)
                noise_seq[0, -2] = noise_item_id # 안전한 아이템 ID 주입

                # [C] 노이즈 주입 후 성능 및 어텐션 측정
                # forward_with_attention은 어텐션 가중치와 로짓을 동시에 반환한다고 가정합니다.
                # BERT4RecWithAttn does not have a forward_with_attention method, but a return_attn parameter
                noise_logits, weights = model(noise_seq, return_attn=True)
                results[group]['noise_hr'].append(calculate_hr(noise_logits, target_batch[i:i+1]))

                # 노이즈 위치(-5)에 대한 마지막 레이어의 평균 어텐션 가중치 추출
                # [layer_idx][batch_idx, head_avg, target_pos, source_pos]
                # weights[-1] has shape [batch_size, seq_len, seq_len]
                attn_1 = weights[-1][0, -1, -1].item()
                attn_2 = weights[-1][0, -1, -2].item()
                noise_attn_val = (attn_1 + attn_2) / 2 # Assuming last item is target and -5 is source
                results[group]['noise_attn'].append(noise_attn_val)

    # 2. 통계 데이터 정리
    stats = []
    for g in ['Q1', 'Q4']:
        if len(results[g]['orig_hr']) > 0:
            stats.append({
                'Group': g,
                'Avg_Noise_Attention': np.mean(results[g]['noise_attn']),
                'HR_Drop': np.mean(results[g]['orig_hr']) - np.mean(results[g]['noise_hr'])
            })

    return pd.DataFrame(stats)

# --- 실행 및 결과 출력 ---
# 런타임 재시작 후 model_attn, test_loader, user_groups가 다시 정의되어 있어야 합니다.
try:
    # Ensure analysis_df has been created and user_groups populated before this call.
    if 'analysis_df' not in globals() or 'user_groups' not in globals():
        print("Error: analysis_df or user_groups not found. Please ensure previous analysis cells are run.")
    else:
        noise_stats_df = analyze_noise_sensitivity_robust(model_attn, test_loader, user_groups, device)

        print("\n" + "="*55)
        print("   [항목 3] 유저 그룹별 노이즈(이질적 장르) 민감도 결과")
        print("="*55)
        print(noise_stats_df.to_string(index=False))
        print("="*55)

        # 시각화
        plt.figure(figsize=(8, 5))
        sns.barplot(data=noise_stats_df, x='Group', y='HR_Drop', palette='magma')
        plt.title('Performance Drop after Noise Injection (Specialist vs Generalist)')
        plt.ylabel('Decrease in Hit Rate')
        plt.show()

except Exception as e:
    print(f"실행 중 오류 발생: {e}")
    print("TIP: 런타임을 재시작했는지, noise_item_id가 Embedding 범위를 넘지 않는지 확인하세요.")